# Event Metadata Validation

## Objective
This notebook validates the `event_metadata.csv` file by checking:

1. Whether transcript timestamps were parsed correctly
2. Whether GMT timestamps convert correctly to `America/New_York`
3. Whether the assigned `event_trading_day` matches the expected rule:
   - if call time is **after 4:00 PM New York time**, use the **next trading day**
   - otherwise use the **same trading day**
4. Whether each `event_trading_day` is an actual trading day



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
EVENT_METADATA_PATH = Path("../data/processed/event_metadata_v1_rawrule.csv")

df = pd.read_csv(EVENT_METADATA_PATH)
df.head()

,ticker,file_name,file_path,call_date_from_filename,year,quarter_label,call_datetime_header_raw,call_datetime_parsed,call_time_gmt,event_trading_day
0,AAPL,2016-Apr-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-04-26,2016,Q2,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26T21:00:00,21:00:00,2016-04-27
1,AAPL,2016-Jan-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-01-26,2016,Q1,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26T22:00:00,22:00:00,2016-01-27
2,AAPL,2016-Jul-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-07-26,2016,Q3,"JULY 26, 2016 / 9:00PM GMT",2016-07-26T21:00:00,21:00:00,2016-07-27
3,AAPL,2016-Oct-25-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-10-25,2016,Q4,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25T21:00:00,21:00:00,2016-10-26
4,AAPL,2017-Aug-01-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2017-08-01,2017,Q3,"AUGUST 01, 2017 / 9:00PM GMT",2017-08-01T21:00:00,21:00:00,2017-08-02


## 1. Parse timestamps and convert to New York time

The `call_datetime_parsed` column is currently stored as a GMT timestamp string.
We convert it to timezone-aware datetime and then convert it to `America/New_York`.

In [3]:
df["call_datetime_gmt"] = pd.to_datetime(df["call_datetime_parsed"], errors="coerce", utc=True)
df["call_datetime_et"] = df["call_datetime_gmt"].dt.tz_convert("America/New_York")

df[[
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "call_datetime_gmt",
    "call_datetime_et"
]].head(10)

,ticker,file_name,call_datetime_header_raw,call_datetime_gmt,call_datetime_et
0,AAPL,2016-Apr-26-AAPL.txt,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26 21:00:00+00:00,2016-04-26 17:00:00-04:00
1,AAPL,2016-Jan-26-AAPL.txt,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26 22:00:00+00:00,2016-01-26 17:00:00-05:00
2,AAPL,2016-Jul-26-AAPL.txt,"JULY 26, 2016 / 9:00PM GMT",2016-07-26 21:00:00+00:00,2016-07-26 17:00:00-04:00
3,AAPL,2016-Oct-25-AAPL.txt,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25 21:00:00+00:00,2016-10-25 17:00:00-04:00
4,AAPL,2017-Aug-01-AAPL.txt,"AUGUST 01, 2017 / 9:00PM GMT",2017-08-01 21:00:00+00:00,2017-08-01 17:00:00-04:00
5,AAPL,2017-Jan-31-AAPL.txt,"JANUARY 31, 2017 / 10:00PM GMT",2017-01-31 22:00:00+00:00,2017-01-31 17:00:00-05:00
6,AAPL,2017-May-02-AAPL.txt,"MAY 02, 2017 / 9:00PM GMT",2017-05-02 21:00:00+00:00,2017-05-02 17:00:00-04:00
7,AAPL,2017-Nov-02-AAPL.txt,"NOVEMBER 02, 2017 / 9:00PM GMT",2017-11-02 21:00:00+00:00,2017-11-02 17:00:00-04:00
8,AAPL,2018-Feb-01-AAPL.txt,"FEBRUARY 01, 2018 / 10:00PM GMT",2018-02-01 22:00:00+00:00,2018-02-01 17:00:00-05:00
9,AAPL,2018-Jul-31-AAPL.txt,"JULY 31, 2018 / 9:00PM GMT",2018-07-31 21:00:00+00:00,2018-07-31 17:00:00-04:00


## 2. Inspect converted times

This helps us verify whether after-hours calls were properly identified.

In [4]:
df["et_date"] = df["call_datetime_et"].dt.date.astype(str)
df["et_time"] = df["call_datetime_et"].dt.strftime("%H:%M:%S")

df[[
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "call_datetime_gmt",
    "call_datetime_et",
    "et_date",
    "et_time",
    "event_trading_day"
]].head(20)

,ticker,file_name,call_datetime_header_raw,call_datetime_gmt,call_datetime_et,et_date,et_time,event_trading_day
0,AAPL,2016-Apr-26-AAPL.txt,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26 21:00:00+00:00,2016-04-26 17:00:00-04:00,2016-04-26,17:00:00,2016-04-27
1,AAPL,2016-Jan-26-AAPL.txt,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26 22:00:00+00:00,2016-01-26 17:00:00-05:00,2016-01-26,17:00:00,2016-01-27
2,AAPL,2016-Jul-26-AAPL.txt,"JULY 26, 2016 / 9:00PM GMT",2016-07-26 21:00:00+00:00,2016-07-26 17:00:00-04:00,2016-07-26,17:00:00,2016-07-27
3,AAPL,2016-Oct-25-AAPL.txt,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25 21:00:00+00:00,2016-10-25 17:00:00-04:00,2016-10-25,17:00:00,2016-10-26
4,AAPL,2017-Aug-01-AAPL.txt,"AUGUST 01, 2017 / 9:00PM GMT",2017-08-01 21:00:00+00:00,2017-08-01 17:00:00-04:00,2017-08-01,17:00:00,2017-08-02
5,AAPL,2017-Jan-31-AAPL.txt,"JANUARY 31, 2017 / 10:00PM GMT",2017-01-31 22:00:00+00:00,2017-01-31 17:00:00-05:00,2017-01-31,17:00:00,2017-02-01
6,AAPL,2017-May-02-AAPL.txt,"MAY 02, 2017 / 9:00PM GMT",2017-05-02 21:00:00+00:00,2017-05-02 17:00:00-04:00,2017-05-02,17:00:00,2017-05-03
7,AAPL,2017-Nov-02-AAPL.txt,"NOVEMBER 02, 2017 / 9:00PM GMT",2017-11-02 21:00:00+00:00,2017-11-02 17:00:00-04:00,2017-11-02,17:00:00,2017-11-03
8,AAPL,2018-Feb-01-AAPL.txt,"FEBRUARY 01, 2018 / 10:00PM GMT",2018-02-01 22:00:00+00:00,2018-02-01 17:00:00-05:00,2018-02-01,17:00:00,2018-02-02
9,AAPL,2018-Jul-31-AAPL.txt,"JULY 31, 2018 / 9:00PM GMT",2018-07-31 21:00:00+00:00,2018-07-31 17:00:00-04:00,2018-07-31,17:00:00,2018-08-01


## 3. Flag calls that happened after market close

Assumption:
- US market close = **4:00 PM America/New_York**
- If call time is **strictly after 4:00 PM**, event day should be the next trading day

In [5]:
df["after_market_close_et"] = (
    (df["call_datetime_et"].dt.hour > 16) |
    ((df["call_datetime_et"].dt.hour == 16) & (df["call_datetime_et"].dt.minute > 0))
)

df[[
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "call_datetime_et",
    "after_market_close_et",
    "event_trading_day"
]].head(20)

,ticker,file_name,call_datetime_header_raw,call_datetime_et,after_market_close_et,event_trading_day
0,AAPL,2016-Apr-26-AAPL.txt,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26 17:00:00-04:00,True,2016-04-27
1,AAPL,2016-Jan-26-AAPL.txt,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26 17:00:00-05:00,True,2016-01-27
2,AAPL,2016-Jul-26-AAPL.txt,"JULY 26, 2016 / 9:00PM GMT",2016-07-26 17:00:00-04:00,True,2016-07-27
3,AAPL,2016-Oct-25-AAPL.txt,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25 17:00:00-04:00,True,2016-10-26
4,AAPL,2017-Aug-01-AAPL.txt,"AUGUST 01, 2017 / 9:00PM GMT",2017-08-01 17:00:00-04:00,True,2017-08-02
5,AAPL,2017-Jan-31-AAPL.txt,"JANUARY 31, 2017 / 10:00PM GMT",2017-01-31 17:00:00-05:00,True,2017-02-01
6,AAPL,2017-May-02-AAPL.txt,"MAY 02, 2017 / 9:00PM GMT",2017-05-02 17:00:00-04:00,True,2017-05-03
7,AAPL,2017-Nov-02-AAPL.txt,"NOVEMBER 02, 2017 / 9:00PM GMT",2017-11-02 17:00:00-04:00,True,2017-11-03
8,AAPL,2018-Feb-01-AAPL.txt,"FEBRUARY 01, 2018 / 10:00PM GMT",2018-02-01 17:00:00-05:00,True,2018-02-02
9,AAPL,2018-Jul-31-AAPL.txt,"JULY 31, 2018 / 9:00PM GMT",2018-07-31 17:00:00-04:00,True,2018-08-01


## 4. Build a trading calendar

We create a set of US business days first.
This is a first-pass approximation and does **not yet include market holidays**.
It is enough for an initial validation notebook.

In [6]:
min_date = pd.to_datetime(df["call_datetime_et"].dt.date.min()) - pd.Timedelta(days=10)
max_date = pd.to_datetime(df["call_datetime_et"].dt.date.max()) + pd.Timedelta(days=10)

business_days = pd.bdate_range(start=min_date, end=max_date)
business_days_set = set(business_days.date)

## 5. Compute expected event trading day

Rule:
- if call happened after 4:00 PM ET, expected event day is the next business day
- otherwise expected event day is the same day if it is a business day
- if same day is not a business day, move to the next business day

In [7]:
def next_business_day(date_value, business_days_index):
    future_days = business_days_index[business_days_index.date > date_value]
    if len(future_days) == 0:
        return pd.NaT
    return future_days[0].date()

def same_or_next_business_day(date_value, business_days_index):
    if date_value in set(business_days_index.date):
        return date_value
    future_days = business_days_index[business_days_index.date > date_value]
    if len(future_days) == 0:
        return pd.NaT
    return future_days[0].date()

def compute_expected_event_day(row, business_days_index):
    if pd.isna(row["call_datetime_et"]):
        return pd.NaT

    call_date = row["call_datetime_et"].date()

    if row["after_market_close_et"]:
        return next_business_day(call_date, business_days_index)
    else:
        return same_or_next_business_day(call_date, business_days_index)

df["expected_event_trading_day"] = df.apply(
    compute_expected_event_day,
    axis=1,
    business_days_index=business_days
)

df["expected_event_trading_day"] = pd.to_datetime(df["expected_event_trading_day"]).dt.date.astype(str)
df[[
    "ticker",
    "file_name",
    "call_datetime_et",
    "after_market_close_et",
    "event_trading_day",
    "expected_event_trading_day"
]].head(20)

,ticker,file_name,call_datetime_et,after_market_close_et,event_trading_day,expected_event_trading_day
0,AAPL,2016-Apr-26-AAPL.txt,2016-04-26 17:00:00-04:00,True,2016-04-27,2016-04-27
1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26 17:00:00-05:00,True,2016-01-27,2016-01-27
2,AAPL,2016-Jul-26-AAPL.txt,2016-07-26 17:00:00-04:00,True,2016-07-27,2016-07-27
3,AAPL,2016-Oct-25-AAPL.txt,2016-10-25 17:00:00-04:00,True,2016-10-26,2016-10-26
4,AAPL,2017-Aug-01-AAPL.txt,2017-08-01 17:00:00-04:00,True,2017-08-02,2017-08-02
5,AAPL,2017-Jan-31-AAPL.txt,2017-01-31 17:00:00-05:00,True,2017-02-01,2017-02-01
6,AAPL,2017-May-02-AAPL.txt,2017-05-02 17:00:00-04:00,True,2017-05-03,2017-05-03
7,AAPL,2017-Nov-02-AAPL.txt,2017-11-02 17:00:00-04:00,True,2017-11-03,2017-11-03
8,AAPL,2018-Feb-01-AAPL.txt,2018-02-01 17:00:00-05:00,True,2018-02-02,2018-02-02
9,AAPL,2018-Jul-31-AAPL.txt,2018-07-31 17:00:00-04:00,True,2018-08-01,2018-08-01


## 6. Compare current vs expected event trading day

In [8]:
df["event_day_matches"] = df["event_trading_day"] == df["expected_event_trading_day"]

match_summary = df["event_day_matches"].value_counts(dropna=False)
match_summary

event_day_matches
True     153
False     35
Name: count, dtype: int64

In [9]:
match_summary_normalized = df["event_day_matches"].value_counts(normalize=True, dropna=False) * 100
match_summary_normalized

event_day_matches
True     81.382979
False    18.617021
Name: proportion, dtype: float64

## 7. Show rows where the assigned event day may be wrong

In [10]:
mismatches = df.loc[~df["event_day_matches"]].copy()

mismatches[[
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "call_datetime_et",
    "after_market_close_et",
    "event_trading_day",
    "expected_event_trading_day"
]].sort_values(["ticker", "file_name"]).head(50)

,ticker,file_name,call_datetime_header_raw,call_datetime_et,after_market_close_et,event_trading_day,expected_event_trading_day
76,CSCO,2016-Aug-17-CSCO.txt,"AUGUST 17, 2016 / 8:30PM GMT",2016-08-17 16:30:00-04:00,True,2016-08-17,2016-08-18
78,CSCO,2016-May-18-CSCO.txt,"MAY 18, 2016 / 8:30PM GMT",2016-05-18 16:30:00-04:00,True,2016-05-18,2016-05-19
80,CSCO,2017-Aug-16-CSCO.txt,"AUGUST 16, 2017 / 8:30PM GMT",2017-08-16 16:30:00-04:00,True,2017-08-16,2017-08-17
82,CSCO,2017-May-17-CSCO.txt,"MAY 17, 2017 / 8:30PM GMT",2017-05-17 16:30:00-04:00,True,2017-05-17,2017-05-18
84,CSCO,2018-Aug-15-CSCO.txt,"AUGUST 15, 2018 / 8:30PM GMT",2018-08-15 16:30:00-04:00,True,2018-08-15,2018-08-16
86,CSCO,2018-May-16-CSCO.txt,"MAY 16, 2018 / 8:30PM GMT",2018-05-16 16:30:00-04:00,True,2018-05-16,2018-05-17
88,CSCO,2019-Aug-14-CSCO.txt,"AUGUST 14, 2019 / 8:30PM GMT",2019-08-14 16:30:00-04:00,True,2019-08-14,2019-08-15
90,CSCO,2019-May-15-CSCO.txt,"MAY 15, 2019 / 8:30PM GMT",2019-05-15 16:30:00-04:00,True,2019-05-15,2019-05-16
92,CSCO,2020-Aug-12-CSCO.txt,"AUGUST 12, 2020 / 8:30PM GMT",2020-08-12 16:30:00-04:00,True,2020-08-12,2020-08-13
94,CSCO,2020-May-13-CSCO.txt,"MAY 13, 2020 / 8:30PM GMT",2020-05-13 16:30:00-04:00,True,2020-05-13,2020-05-14


## 8. Focus on likely problem times

Rows around `20:30 GMT`, `21:00 GMT`, and `21:30 GMT` are especially important because daylight saving time affects whether they are before or after 4:00 PM ET.

In [11]:
df["gmt_hour_minute"] = df["call_datetime_gmt"].dt.strftime("%H:%M")
problem_times = df[df["gmt_hour_minute"].isin(["20:30", "21:00", "21:30", "22:00", "22:30"])]
display_cols = [
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "call_datetime_gmt",
    "call_datetime_et",
    "after_market_close_et",
    "event_trading_day",
    "expected_event_trading_day",
    "event_day_matches",
]
problem_times.sort_values(["gmt_hour_minute", "ticker"])[display_cols].head(100)


,ticker,file_name,call_datetime_header_raw,call_datetime_gmt,call_datetime_et,after_market_close_et,event_trading_day,expected_event_trading_day,event_day_matches
76,CSCO,2016-Aug-17-CSCO.txt,"AUGUST 17, 2016 / 8:30PM GMT",2016-08-17 20:30:00+00:00,2016-08-17 16:30:00-04:00,True,2016-08-17,2016-08-18,False
78,CSCO,2016-May-18-CSCO.txt,"MAY 18, 2016 / 8:30PM GMT",2016-05-18 20:30:00+00:00,2016-05-18 16:30:00-04:00,True,2016-05-18,2016-05-19,False
80,CSCO,2017-Aug-16-CSCO.txt,"AUGUST 16, 2017 / 8:30PM GMT",2017-08-16 20:30:00+00:00,2017-08-16 16:30:00-04:00,True,2017-08-16,2017-08-17,False
82,CSCO,2017-May-17-CSCO.txt,"MAY 17, 2017 / 8:30PM GMT",2017-05-17 20:30:00+00:00,2017-05-17 16:30:00-04:00,True,2017-05-17,2017-05-18,False
84,CSCO,2018-Aug-15-CSCO.txt,"AUGUST 15, 2018 / 8:30PM GMT",2018-08-15 20:30:00+00:00,2018-08-15 16:30:00-04:00,True,2018-08-15,2018-08-16,False
...,...,...,...,...,...,...,...,...,...
53,AMZN,2019-Oct-24-AMZN.txt,"OCTOBER 24, 2019 / 9:30PM GMT",2019-10-24 21:30:00+00:00,2019-10-24 17:30:00-04:00,True,2019-10-25,2019-10-25,True
54,AMZN,2020-Apr-30-AMZN.txt,"APRIL 30, 2020 / 9:30PM GMT",2020-04-30 21:30:00+00:00,2020-04-30 17:30:00-04:00,True,2020-05-01,2020-05-01,True
56,AMZN,2020-Jul-30-AMZN.txt,"JULY 30, 2020 / 9:30PM GMT",2020-07-30 21:30:00+00:00,2020-07-30 17:30:00-04:00,True,2020-07-31,2020-07-31,True
77,CSCO,2016-Feb-10-CSCO.txt,"FEBRUARY 10, 2016 / 9:30PM GMT",2016-02-10 21:30:00+00:00,2016-02-10 16:30:00-05:00,True,2016-02-11,2016-02-11,True


## 9. Manual audit sample

Create a small validation table for hand-checking a subset of rows.

In [12]:
audit_sample = df.sample(25, random_state=42)[[
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "call_datetime_et",
    "after_market_close_et",
    "event_trading_day",
    "expected_event_trading_day",
    "event_day_matches"
]].sort_values(["ticker", "file_name"])

audit_sample

,ticker,file_name,call_datetime_header_raw,call_datetime_et,after_market_close_et,event_trading_day,expected_event_trading_day,event_day_matches
15,AAPL,2019-Oct-30-AAPL.txt,"OCTOBER 30, 2019 / 9:00PM GMT",2019-10-30 17:00:00-04:00,True,2019-10-31,2019-10-31,True
16,AAPL,2020-Apr-30-AAPL.txt,"APRIL 30, 2020 / 9:00PM GMT",2020-04-30 17:00:00-04:00,True,2020-05-01,2020-05-01,True
18,AAPL,2020-Jul-30-AAPL.txt,"JULY 30, 2020 / 9:00PM GMT",2020-07-30 17:00:00-04:00,True,2020-07-31,2020-07-31,True
19,AMD,2016-Apr-21-AMD.txt,"APRIL 21, 2016 / 9:00PM GMT",2016-04-21 17:00:00-04:00,True,2016-04-22,2016-04-22,True
30,AMD,2018-Oct-24-AMD.txt,"OCTOBER 24, 2018 / 9:30PM GMT",2018-10-24 17:30:00-04:00,True,2018-10-25,2018-10-25,True
42,AMZN,2017-Apr-27-AMZN.txt,"APRIL 27, 2017 / 9:30PM GMT",2017-04-27 17:30:00-04:00,True,2017-04-28,2017-04-28,True
45,AMZN,2017-Oct-26-AMZN.txt,"OCTOBER 26, 2017 / 9:30PM GMT",2017-10-26 17:30:00-04:00,True,2017-10-27,2017-10-27,True
51,AMZN,2019-Jan-31-AMZN.txt,"JANUARY 31, 2019 / 10:30PM GMT",2019-01-31 17:30:00-05:00,True,2019-02-01,2019-02-01,True
55,AMZN,2020-Jan-30-AMZN.txt,"JANUARY 30, 2020 / 10:30PM GMT",2020-01-30 17:30:00-05:00,True,2020-01-31,2020-01-31,True
60,ASML,2016-Oct-19-ASML.txt,"OCTOBER 19, 2016 / 1:00PM GMT",2016-10-19 09:00:00-04:00,False,2016-10-19,2016-10-19,True


## 10. Check whether event_trading_day is a business day

This confirms whether assigned event days fall on weekdays.
Again, this does not yet account for market holidays.

In [13]:
df["event_trading_day_dt"] = pd.to_datetime(df["event_trading_day"], errors="coerce")
df["event_day_is_business_day"] = df["event_trading_day_dt"].dt.date.isin(business_days_set)

df["event_day_is_business_day"].value_counts(dropna=False)

event_day_is_business_day
True    188
Name: count, dtype: int64

In [14]:
non_business_days = df.loc[~df["event_day_is_business_day"]].copy()

non_business_days[[
    "ticker",
    "file_name",
    "call_datetime_header_raw",
    "event_trading_day"
]].head(50)

,ticker,file_name,call_datetime_header_raw,event_trading_day


## 11. Summary diagnostics

In [15]:
summary = {
    "total_rows": len(df),
    "rows_with_valid_gmt_timestamp": int(df["call_datetime_gmt"].notna().sum()),
    "rows_matching_expected_event_day": int(df["event_day_matches"].sum()),
    "rows_not_matching_expected_event_day": int((~df["event_day_matches"]).sum()),
    "rows_with_business_day_event_date": int(df["event_day_is_business_day"].sum()),
    "rows_with_non_business_day_event_date": int((~df["event_day_is_business_day"]).sum()),
}

summary

{'total_rows': 188,
 'rows_with_valid_gmt_timestamp': 188,
 'rows_matching_expected_event_day': 153,
 'rows_not_matching_expected_event_day': 35,
 'rows_with_business_day_event_date': 188,
 'rows_with_non_business_day_event_date': 0}

The validation process confirmed that all transcript timestamps were successfully parsed (188/188 valid timestamps).
However, the initial event-day assignment rule produced incorrect results for 35 observations (18.6%) due to ignoring daylight saving time when converting GMT timestamps to US/Eastern time.

Specifically, earnings calls occurring at 20:30 GMT correspond to 16:30 Eastern Time during daylight saving periods, which is after the US market close (4:00 PM ET). These events should therefore be assigned to the next trading day, whereas the initial rule incorrectly assigned them to the same day.

After correcting the event-day logic using timezone-aware conversion to America/New_York, the expected event trading days were recomputed and used as the final event alignment variable.